# ViCAM - Viral fine-tuning of Cambriam

In [7]:
import os
from Bio import SeqIO

In [ ]:
seqkit rmdup -s -j 28 data/raw/URVDBv30prot.fasta > data/raw/URVDBv30prot_rmdup.fasta
total: 
remained: 5535631

In [ ]:
python src/data/preprocess.py
Total sequences: 5535631
Valid sequences: 2696018
2007048 sequences were longer than 1600.

Eucariote sequences filter in uniprot
(taxonomy_id:2759) AND (reviewed:true) AND (Length:[* TO 1600])

In [5]:
#input_fasta = "../data/raw/URVDBv29-prot_clustered.fasta"
input_fasta = "../data/processed/C-RVDBv29_no_poly/train.fasta"

records = list(SeqIO.parse(input_fasta, "fasta"))

#records = [record for record in records if 'poly' in record.description]

In [6]:
len(records)

568485

In [7]:
mn=0
mx=0
for record in records:
    if len(record.seq) < mn or mn == 0:
        mn = len(record.seq)
    if len(record.seq) > mx:
        mx = len(record.seq)
print(f"Min length: {mn}")
print(f"Max length: {mx}")


Min length: 11
Max length: 2047


In [ ]:
count = 0
while count < 10:
    record = records[count]
    print(record.description)
    count += 1

acc|GENBANK|UJJ65121.1|GENBANK|OM336640|surface glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65122.1|GENBANK|OM336640|ORF3a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65123.1|GENBANK|OM336640|envelope protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65124.1|GENBANK|OM336640|membrane glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65125.1|GENBANK|OM336640|ORF6 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65126.1|GENBANK|OM336640|ORF7a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65127.1|GENBANK|OM336640|ORF7b [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65128.1|GENBANK|OM336640|ORF8 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65129.1|GENBANK|OM336640|nucleocapsid phosphoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65130.1|GENBANK|OM336640|OR

In [7]:
data_dir = '../data/viral/mutant_sequences'

for file in os.listdir(data_dir):
    records = list(SeqIO.parse(os.path.join(data_dir, file), "fasta"))
    if len(records[0].seq) > 1022:
        print(records[0].id, len(records[0].seq))

    

SARS2_RBD_N331C 1273
CVB3_POLG_M1D 2185
EV_REP_G1I 1331
SARS2_DELTA_M1F 1250
SARS2_RBD_N331C 1273
SARS2_BA1_M1I 1249


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def filter_fasta_by_valid_amino_acids(input_fasta_path, output_fasta_path):
    """
    Filters a FASTA file to keep only sequences with valid amino acids.

    Args:
        input_fasta_path (str): Path to the input FASTA file.
        output_fasta_path (str): Path to save the filtered FASTA file.
    """
    valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
    
    valid_records = []
    for record in SeqIO.parse(input_fasta_path, "fasta"):
        sequence_str = str(record.seq).upper()
        is_valid = True
        for aa in sequence_str:
            if aa not in valid_amino_acids:
                is_valid = False
                print(f"Invalid character '{aa}' found in sequence: {record.id}. Skipping.")
                break
        if is_valid:
            valid_records.append(record)

    SeqIO.write(valid_records, output_fasta_path, "fasta")
    print(f"Filtered FASTA saved to: {output_fasta_path}")
    print(f"Original records: {len(list(SeqIO.parse(input_fasta_path, 'fasta')))}, Valid records: {len(valid_records)}")

# Example usage:
input_file = "../data/processed/C-RVDBv29_no_poly/train.fasta"
output_file = "../data/processed/C-RVDBv29_no_poly_20aa/train.fasta"
filter_fasta_by_valid_amino_acids(input_file, output_file)

In [4]:
[x for x in ['vicam_300m', 'vicam_600m','esmc_300m', 'esmc_600m'] if x in '/chpc/home/C-RVDBv29_no_poly/vicam_600m.fasta'][0]

'vicam_600m'

# Creating mutant fasta file

In [1]:
import os
import pandas as pd

In [27]:
base_dir = '../data/viral/metadata/'
for file in os.listdir(base_dir):
    df = pd.read_csv(os.path.join(base_dir, file))
    df['ID'] = df['ID'].apply(lambda x: '_'.join(x.split('_')[:-1])) + '_' + df['mutant']
    df.to_csv(os.path.join(base_dir, file), index=False)

df

,ID,mutant,num_mutations,target,sequence
0,SARS2_RBD_N331C,N331C,1,-1.26,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
1,SARS2_RBD_N331D,N331D,1,-0.44,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
2,SARS2_RBD_N331A,N331A,1,-0.11,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3,SARS2_RBD_N331Y,N331Y,1,-1.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
4,SARS2_RBD_N331W,N331W,1,-1.12,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
...,...,...,...,...,...
3793,SARS2_RBD_T531F,T531F,1,-0.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3794,SARS2_RBD_T531E,T531E,1,0.03,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3795,SARS2_RBD_T531D,T531D,1,0.03,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3796,SARS2_RBD_T531A,T531A,1,-0.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [28]:
base_dir = '../data/viral/metadata/'
for file in os.listdir(base_dir):
    df = pd.read_csv(os.path.join(base_dir, file))
    file_name = file.strip('.csv')

    with open(f"../data/viral/mutant_sequences/{file_name}.fasta", "w") as f:
        for _, row in df.iterrows():
            f.write(f">{row['ID']}\n{row['sequence']}\n")


# Testing model embeddings

In [1]:
import torch
import pandas as pd
import numpy as np

from scipy import stats
from sklearn import metrics
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings('ignore') 
from sklearn.exceptions import ConvergenceWarning

In [42]:
def features_scaler(features):
    '''Scale the features by min-max scaler, to ensure that the features selected by Lasso are not biased by the scale of the features'''
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_features = scaler.fit_transform(features)
    return pd.DataFrame(scaled_features)


def run_regression(features, target):
    '''this version computes y_pred for train and test sets'''
    # Initialize lists for storing results
    folds, num_nonzero_coefs = [], []
    r2s_train, maes_train, rmses_train = [], [], []
    r2s_test, maes_test, rmses_test = [], [], []
    rhos_train, rhos_test = [], []

    # Define the KFold cross-validator
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # Loop over the KFold splits
    for kfold, (train_index, test_index) in enumerate(kf.split(features)):
        # Split the data into training and testing sets
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        # Define and train the regression model
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ConvergenceWarning)
            model = LassoCV(max_iter=1000, tol=1e-2, n_jobs=-1)
            model.fit(X_train, y_train)

            # get the number of non-zero coefficients
            coeficients = model.coef_
            num_nonzero_coef = np.sum(coeficients != 0)

            # Make predictions
            y_pred_train = pd.DataFrame(model.predict(X_train))
            y_pred_test = pd.DataFrame(model.predict(X_test))

            # Evaluate the model
            r2_train = metrics.r2_score(y_train, y_pred_train)
            mae_train = metrics.mean_absolute_error(y_train, y_pred_train)
            mse_train = metrics.mean_squared_error(y_train, y_pred_train)
            rmse_train = np.sqrt(mse_train)
            rho_train, p_value_train = spearmanr(y_train, y_pred_train)

            r2_test = metrics.r2_score(y_test, y_pred_test)
            mae_test = metrics.mean_absolute_error(y_test, y_pred_test)
            mse_test = metrics.mean_squared_error(y_test, y_pred_test)
            rmse_test = np.sqrt(mse_test)
            rho_test, p_value_test = spearmanr(y_test, y_pred_test)

            # Append results
            r2s_train.append(r2_train)
            maes_train.append(mae_train)
            rmses_train.append(rmse_train)
            rhos_train.append(rho_train)

            r2s_test.append(r2_test)
            maes_test.append(mae_test)
            rmses_test.append(rmse_test)
            rhos_test.append(rho_test)


            folds.append(kfold + 1)
            num_nonzero_coefs.append(num_nonzero_coef)

        # Return the collected results
        print(f"Results:  fold {kfold}, r2_train: {r2_train:.3f}, r2_test: {r2_test:.3f}, Num coefs: {num_nonzero_coef}")
    print(f"Results:  r2_train: {np.mean(r2s_train):.2f}, r2_test: {np.mean(r2s_test):.2f}, Num coefs: {np.mean(num_nonzero_coefs):.2f}")
    return r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs


## ESM C 300M

In [4]:
metadata = pd.read_csv("../data/DMS_mut_metadata/PA_FLU_Sun2015_metadata.csv", index_col=0)  

embeddings = pd.DataFrame(torch.load('../embeddings/esmc_300m/PA_FLU_Sun2015_embeddings.pt')).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)


data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(2590, 960)
(2590,)


,ID,mutant,target,sequence,0,1,2,3,4,5,...,950,951,952,953,954,955,956,957,958,959
0,PA_FLU_C8R,C8R,0.120065,MEDFVRQRFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001842,0.001954,0.010462,0.012958,0.003278,0.003426,...,0.000652,-0.012437,-0.007702,0.002752,-0.004459,0.009886,0.003699,-0.003755,-0.006035,-0.007248
1,PA_FLU_C8Y,C8Y,0.869537,MEDFVRQYFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001988,0.002019,0.010245,0.012934,0.003225,0.003475,...,0.000601,-0.012689,-0.007409,0.002945,-0.004173,0.009672,0.003122,-0.003733,-0.006044,-0.007270
2,PA_FLU_C8C,C8C,0.426665,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
3,PA_FLU_F9L,F9L,0.119464,MEDFVRQCLNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001603,0.001408,0.010345,0.012646,0.003834,0.003286,...,0.000857,-0.012337,-0.006629,0.002766,-0.004546,0.010311,0.003645,-0.004141,-0.006735,-0.007024
4,PA_FLU_F9S,F9S,0.078119,MEDFVRQCSNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001810,0.001537,0.010888,0.012893,0.003228,0.003209,...,0.000773,-0.012788,-0.006652,0.002718,-0.004526,0.010168,0.003424,-0.003940,-0.006996,-0.007344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,PA_FLU_L715L,L715L,1.394932,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
2586,PA_FLU_L715S,L715S,0.914543,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001683,0.001725,0.010480,0.012886,0.003553,0.003232,...,0.000785,-0.012572,-0.006956,0.002753,-0.004591,0.010266,0.003608,-0.004101,-0.006722,-0.007024
2587,PA_FLU_L715L,L715L,0.446139,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001767,0.001509,0.010462,0.012814,0.003647,0.003377,...,0.000815,-0.012426,-0.006838,0.002727,-0.004453,0.010092,0.003475,-0.004122,-0.006680,-0.007045
2588,PA_FLU_R716G,R716G,0.777810,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,0.001630,0.001480,0.010464,0.013056,0.003677,0.003463,...,0.000961,-0.012293,-0.006662,0.002908,-0.004706,0.010195,0.003459,-0.004245,-0.006762,-0.007140


In [5]:
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

Results:  fold 0, r2_train: 0.042, r2_test: 0.048, Num coefs: 20
Results:  fold 1, r2_train: 0.024, r2_test: 0.009, Num coefs: 13
Results:  fold 2, r2_train: 0.057, r2_test: 0.019, Num coefs: 23
Results:  fold 3, r2_train: 0.055, r2_test: 0.020, Num coefs: 17
Results:  fold 4, r2_train: 0.039, r2_test: -0.004, Num coefs: 17
Results:  r2_train: 0.04, r2_test: 0.02, Num coefs: 18.00


## ViCam 300M

In [10]:
metadata = pd.read_csv("../data/DMS_mut_metadata/PA_FLU_Sun2015_metadata.csv", index_col=0)  

#embeddings = pd.DataFrame(torch.load('../embeddings/vicam_300m/CRVDBv29_maxLen2046_20aa_Full_lr1e6/PA_FLU_Sun2015_embeddings.pt', map_location=torch.device('cpu'))).T
#embeddings = pd.DataFrame(torch.load('../embeddings/vicam_300m/CRVDBv29_maxLen2046_20aa_Full_RLRP_lr1e6/PA_FLU_Sun2015_embeddings.pt', map_location=torch.device('cpu'))).T
embeddings = pd.DataFrame(torch.load('../embeddings/ViCAM/CRVDBv29_maxLen1022_Full_lr5e4_RLRP/PA_FLU_Sun2015_embeddings.pt', map_location=torch.device('cpu'))).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)


data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(2590, 960)
(2590,)


,ID,mutant,target,sequence,0,1,2,3,4,5,...,950,951,952,953,954,955,956,957,958,959
0,PA_FLU_C8R,C8R,0.120065,MEDFVRQRFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038859,0.006309,0.020087,0.005971,-0.009546,0.035468,...,0.052569,-0.023458,0.037337,-0.018110,0.004505,-0.016892,0.039747,0.006908,-0.007801,-0.030174
1,PA_FLU_C8Y,C8Y,0.869537,MEDFVRQYFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038784,0.006345,0.020044,0.005974,-0.009484,0.035487,...,0.052643,-0.023479,0.037390,-0.017988,0.004495,-0.016904,0.039488,0.006976,-0.007973,-0.030108
2,PA_FLU_C8C,C8C,0.426665,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038842,0.006220,0.020181,0.005939,-0.009549,0.035511,...,0.052703,-0.023621,0.037423,-0.018095,0.004391,-0.016925,0.039693,0.006982,-0.008038,-0.030183
3,PA_FLU_F9L,F9L,0.119464,MEDFVRQCLNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038728,0.006134,0.020096,0.005920,-0.009511,0.035415,...,0.052524,-0.023571,0.037328,-0.018008,0.004503,-0.016927,0.039515,0.006954,-0.007960,-0.030281
4,PA_FLU_F9S,F9S,0.078119,MEDFVRQCSNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038916,0.006052,0.020101,0.006033,-0.009621,0.035555,...,0.052667,-0.023636,0.037234,-0.017964,0.004566,-0.016970,0.039677,0.006914,-0.007980,-0.030213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,PA_FLU_L715L,L715L,1.394932,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038842,0.006220,0.020181,0.005939,-0.009549,0.035511,...,0.052703,-0.023621,0.037423,-0.018095,0.004391,-0.016925,0.039693,0.006982,-0.008038,-0.030183
2586,PA_FLU_L715S,L715S,0.914543,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038669,0.006329,0.020157,0.005997,-0.009423,0.035486,...,0.052986,-0.023883,0.037749,-0.018173,0.004526,-0.017303,0.039752,0.006908,-0.008113,-0.030121
2587,PA_FLU_L715L,L715L,0.446139,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038842,0.006220,0.020181,0.005939,-0.009549,0.035511,...,0.052703,-0.023621,0.037423,-0.018095,0.004391,-0.016925,0.039693,0.006982,-0.008038,-0.030183
2588,PA_FLU_R716G,R716G,0.777810,MEDFVRQCFNPMIVELAEKAMKEYGEDLKIETNKFAAICTHLEVCF...,-0.038438,0.006380,0.020031,0.005780,-0.009570,0.035687,...,0.053206,-0.023586,0.038041,-0.018390,0.004249,-0.017290,0.039863,0.007154,-0.008288,-0.030514


In [7]:
# CRVDBv29_maxLen2046_20aa_Full_lr1e6
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

Results:  fold 0, r2_train: 0.187, r2_test: 0.207, Num coefs: 30
Results:  fold 1, r2_train: 0.196, r2_test: 0.179, Num coefs: 32
Results:  fold 2, r2_train: 0.180, r2_test: 0.253, Num coefs: 35
Results:  fold 3, r2_train: 0.218, r2_test: 0.132, Num coefs: 42
Results:  fold 4, r2_train: 0.206, r2_test: 0.105, Num coefs: 37
Results:  r2_train: 0.20, r2_test: 0.18, Num coefs: 35.20


In [9]:
# CRVDBv29_maxLen2046_20aa_Full_RLRP_lr1e6
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

Results:  fold 0, r2_train: 0.158, r2_test: 0.183, Num coefs: 13
Results:  fold 1, r2_train: 0.203, r2_test: 0.192, Num coefs: 55
Results:  fold 2, r2_train: 0.147, r2_test: 0.224, Num coefs: 13
Results:  fold 3, r2_train: 0.217, r2_test: 0.133, Num coefs: 42
Results:  fold 4, r2_train: 0.210, r2_test: 0.107, Num coefs: 34
Results:  r2_train: 0.19, r2_test: 0.17, Num coefs: 31.40


In [11]:
# CRVDBv29_maxLen1022_Full_lr5e4_RLRP
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

Results:  fold 0, r2_train: 0.240, r2_test: 0.234, Num coefs: 61
Results:  fold 1, r2_train: 0.271, r2_test: 0.214, Num coefs: 85
Results:  fold 2, r2_train: 0.247, r2_test: 0.248, Num coefs: 67
Results:  fold 3, r2_train: 0.285, r2_test: 0.193, Num coefs: 84
Results:  fold 4, r2_train: 0.285, r2_test: 0.155, Num coefs: 85
Results:  r2_train: 0.27, r2_test: 0.21, Num coefs: 76.40


# rsawhney esm2_3B

In [37]:
metadata = pd.read_csv("../data/viral/metadata/LASSA_GP_Carr.csv")  
metadata

,ID,mutant,num_mutations,target,sequence
0,LASSA_GP_M1Y,M1Y,1,-3.50700,YGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
1,LASSA_GP_M1W,M1W,1,-3.13900,WGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
2,LASSA_GP_M1V,M1V,1,-3.63700,VGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
3,LASSA_GP_M1T,M1T,1,-3.88400,TGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
4,LASSA_GP_M1S,M1S,1,-4.14600,SGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
...,...,...,...,...,...
9270,LASSA_GP_R491E,R491E,1,-1.69200,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
9271,LASSA_GP_R491D,R491D,1,-1.44500,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
9272,LASSA_GP_R491C,R491C,1,0.07979,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
9273,LASSA_GP_R491A,R491A,1,-0.41640,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...


In [38]:
embeddings = pd.DataFrame(torch.load('../embeddings/rsawhney_esm2_3B/viral/LASSA_GP_Carr.pt')).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)
embeddings

,ID,0,1,2,3,4,5,6,7,8,...,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559
0,LASSA_GP_M1Y,-0.023741,-0.014115,-0.004650,-0.027688,-0.016437,0.000481,-0.025619,-0.015014,-0.047618,...,-0.001048,-0.052441,-0.009433,-0.026791,-0.014866,0.057837,-0.039273,-0.005322,-0.017274,-0.036605
1,LASSA_GP_M1W,-0.023723,-0.014224,-0.004792,-0.027722,-0.016564,0.000298,-0.025604,-0.015075,-0.047758,...,-0.001001,-0.052488,-0.009497,-0.026858,-0.014892,0.057558,-0.039195,-0.005383,-0.017377,-0.036769
2,LASSA_GP_M1V,-0.023737,-0.014303,-0.004858,-0.027681,-0.016394,0.000580,-0.025559,-0.014913,-0.047745,...,-0.001019,-0.052438,-0.009334,-0.026773,-0.014893,0.057486,-0.039523,-0.005396,-0.017307,-0.036933
3,LASSA_GP_M1T,-0.023738,-0.014309,-0.004467,-0.027674,-0.016195,0.000472,-0.025603,-0.014744,-0.047542,...,-0.001018,-0.052457,-0.009127,-0.026923,-0.014854,0.057582,-0.039092,-0.005276,-0.017216,-0.036879
4,LASSA_GP_M1S,-0.023735,-0.014382,-0.005035,-0.027690,-0.016572,0.000295,-0.025556,-0.014973,-0.047515,...,-0.001029,-0.052452,-0.009533,-0.026817,-0.014889,0.057538,-0.039367,-0.005143,-0.017330,-0.036872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9270,LASSA_GP_R491E,-0.023754,-0.013520,-0.004783,-0.027839,-0.016745,0.000666,-0.025541,-0.015140,-0.047321,...,-0.001015,-0.053056,-0.009807,-0.026642,-0.014630,0.058438,-0.039567,-0.005205,-0.017071,-0.036969
9271,LASSA_GP_R491D,-0.023755,-0.014091,-0.004623,-0.027836,-0.016841,0.000598,-0.025666,-0.015094,-0.047851,...,-0.001044,-0.053082,-0.009741,-0.026593,-0.014632,0.058507,-0.039570,-0.005281,-0.017105,-0.037001
9272,LASSA_GP_R491C,-0.024725,-0.013618,-0.004396,-0.027876,-0.016836,0.000450,-0.025520,-0.014670,-0.047773,...,-0.001035,-0.053138,-0.009594,-0.026753,-0.014611,0.058401,-0.039391,-0.004830,-0.016973,-0.037096
9273,LASSA_GP_R491A,-0.023756,-0.014002,-0.004701,-0.027888,-0.017043,0.000416,-0.025565,-0.015142,-0.047609,...,-0.000995,-0.053071,-0.009805,-0.026534,-0.014722,0.058450,-0.039568,-0.005668,-0.017157,-0.037035


In [ ]:
data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
data

,ID,mutant,num_mutations,target,sequence,0,1,2,3,4,...,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559
0,LASSA_GP_M1Y,M1Y,1,-3.50700,YGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023741,-0.014115,-0.004650,-0.027688,-0.016437,...,-0.001048,-0.052441,-0.009433,-0.026791,-0.014866,0.057837,-0.039273,-0.005322,-0.017274,-0.036605
1,LASSA_GP_M1W,M1W,1,-3.13900,WGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023723,-0.014224,-0.004792,-0.027722,-0.016564,...,-0.001001,-0.052488,-0.009497,-0.026858,-0.014892,0.057558,-0.039195,-0.005383,-0.017377,-0.036769
2,LASSA_GP_M1V,M1V,1,-3.63700,VGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023737,-0.014303,-0.004858,-0.027681,-0.016394,...,-0.001019,-0.052438,-0.009334,-0.026773,-0.014893,0.057486,-0.039523,-0.005396,-0.017307,-0.036933
3,LASSA_GP_M1T,M1T,1,-3.88400,TGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023738,-0.014309,-0.004467,-0.027674,-0.016195,...,-0.001018,-0.052457,-0.009127,-0.026923,-0.014854,0.057582,-0.039092,-0.005276,-0.017216,-0.036879
4,LASSA_GP_M1S,M1S,1,-4.14600,SGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023735,-0.014382,-0.005035,-0.027690,-0.016572,...,-0.001029,-0.052452,-0.009533,-0.026817,-0.014889,0.057538,-0.039367,-0.005143,-0.017330,-0.036872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9270,LASSA_GP_R491E,R491E,1,-1.69200,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023754,-0.013520,-0.004783,-0.027839,-0.016745,...,-0.001015,-0.053056,-0.009807,-0.026642,-0.014630,0.058438,-0.039567,-0.005205,-0.017071,-0.036969
9271,LASSA_GP_R491D,R491D,1,-1.44500,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023755,-0.014091,-0.004623,-0.027836,-0.016841,...,-0.001044,-0.053082,-0.009741,-0.026593,-0.014632,0.058507,-0.039570,-0.005281,-0.017105,-0.037001
9272,LASSA_GP_R491C,R491C,1,0.07979,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.024725,-0.013618,-0.004396,-0.027876,-0.016836,...,-0.001035,-0.053138,-0.009594,-0.026753,-0.014611,0.058401,-0.039391,-0.004830,-0.016973,-0.037096
9273,LASSA_GP_R491A,R491A,1,-0.41640,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023756,-0.014002,-0.004701,-0.027888,-0.017043,...,-0.000995,-0.053071,-0.009805,-0.026534,-0.014722,0.058450,-0.039568,-0.005668,-0.017157,-0.037035


In [43]:
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(9275, 2560)
(9275,)


,ID,mutant,num_mutations,target,sequence,0,1,2,3,4,...,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559
0,LASSA_GP_M1Y,M1Y,1,-3.50700,YGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023741,-0.014115,-0.004650,-0.027688,-0.016437,...,-0.001048,-0.052441,-0.009433,-0.026791,-0.014866,0.057837,-0.039273,-0.005322,-0.017274,-0.036605
1,LASSA_GP_M1W,M1W,1,-3.13900,WGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023723,-0.014224,-0.004792,-0.027722,-0.016564,...,-0.001001,-0.052488,-0.009497,-0.026858,-0.014892,0.057558,-0.039195,-0.005383,-0.017377,-0.036769
2,LASSA_GP_M1V,M1V,1,-3.63700,VGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023737,-0.014303,-0.004858,-0.027681,-0.016394,...,-0.001019,-0.052438,-0.009334,-0.026773,-0.014893,0.057486,-0.039523,-0.005396,-0.017307,-0.036933
3,LASSA_GP_M1T,M1T,1,-3.88400,TGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023738,-0.014309,-0.004467,-0.027674,-0.016195,...,-0.001018,-0.052457,-0.009127,-0.026923,-0.014854,0.057582,-0.039092,-0.005276,-0.017216,-0.036879
4,LASSA_GP_M1S,M1S,1,-4.14600,SGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023735,-0.014382,-0.005035,-0.027690,-0.016572,...,-0.001029,-0.052452,-0.009533,-0.026817,-0.014889,0.057538,-0.039367,-0.005143,-0.017330,-0.036872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9270,LASSA_GP_R491E,R491E,1,-1.69200,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023754,-0.013520,-0.004783,-0.027839,-0.016745,...,-0.001015,-0.053056,-0.009807,-0.026642,-0.014630,0.058438,-0.039567,-0.005205,-0.017071,-0.036969
9271,LASSA_GP_R491D,R491D,1,-1.44500,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023755,-0.014091,-0.004623,-0.027836,-0.016841,...,-0.001044,-0.053082,-0.009741,-0.026593,-0.014632,0.058507,-0.039570,-0.005281,-0.017105,-0.037001
9272,LASSA_GP_R491C,R491C,1,0.07979,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.024725,-0.013618,-0.004396,-0.027876,-0.016836,...,-0.001035,-0.053138,-0.009594,-0.026753,-0.014611,0.058401,-0.039391,-0.004830,-0.016973,-0.037096
9273,LASSA_GP_R491A,R491A,1,-0.41640,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...,-0.023756,-0.014002,-0.004701,-0.027888,-0.017043,...,-0.000995,-0.053071,-0.009805,-0.026534,-0.014722,0.058450,-0.039568,-0.005668,-0.017157,-0.037035


In [45]:
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

Results:  fold 0, r2_train: 0.072, r2_test: 0.075, Num coefs: 13
Results:  fold 1, r2_train: 0.074, r2_test: 0.071, Num coefs: 15
Results:  fold 2, r2_train: 0.076, r2_test: 0.067, Num coefs: 15
Results:  fold 3, r2_train: 0.073, r2_test: 0.076, Num coefs: 12
Results:  fold 4, r2_train: 0.076, r2_test: 0.061, Num coefs: 13
Results:  r2_train: 0.07, r2_test: 0.07, Num coefs: 13.60


# HG FLU

In [9]:
metadata = pd.read_csv("../data/DMS_metadata/HG_FLU_Bloom2016_metadata.csv", index_col=0)  

embeddings = pd.DataFrame(torch.load('../embeddings/ViCAM/CRVDBv29_maxLen2046_20aa_Full_lr1e6/HG_FLU_Bloom2016_embeddings.pt')).T
embeddings.reset_index(inplace=True)
embeddings.rename(columns={"index": "ID"}, inplace=True)


data = metadata.merge(embeddings, how='inner', left_on='ID', right_on='ID')
target = data['target']
features = data.iloc[:, metadata.shape[1]:]
features = features_scaler(features)
print(features.shape)
print(target.shape)
data

(11279, 960)
(11279,)


,ID,mutant,target,sequence,0,1,2,3,4,5,...,950,951,952,953,954,955,956,957,958,959
0,HG_FLU_K2C,K2C,0.224238,MCAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005097,-0.021731,0.010277,-0.013421,-0.008552,0.000414,...,0.012412,-0.006495,0.009853,-0.005803,0.010278,-0.004014,0.001049,0.021961,0.001836,0.011919
1,HG_FLU_K2E,K2E,-1.005736,MEAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005384,-0.021722,0.010468,-0.013126,-0.008439,0.000633,...,0.012529,-0.006767,0.010149,-0.005761,0.010055,-0.003889,0.001138,0.021689,0.001743,0.011635
2,HG_FLU_K2D,K2D,-1.858330,MDAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005207,-0.021604,0.010262,-0.013415,-0.008443,0.000627,...,0.012385,-0.006650,0.009913,-0.005562,0.010081,-0.003743,0.001195,0.021784,0.001754,0.011595
3,HG_FLU_K2G,K2G,-1.005611,MGAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005312,-0.021917,0.010379,-0.013276,-0.008483,0.000674,...,0.012640,-0.006699,0.010067,-0.005755,0.010160,-0.003899,0.001047,0.021828,0.001740,0.011945
4,HG_FLU_K2F,K2F,0.275228,MFAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.004913,-0.021450,0.010101,-0.013711,-0.008718,0.000135,...,0.012213,-0.006356,0.009588,-0.005782,0.010607,-0.004116,0.001082,0.021898,0.002101,0.011659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11274,HG_FLU_I565R,I565R,-3.828152,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005479,-0.022010,0.009686,-0.013076,-0.008462,0.000590,...,0.012307,-0.006155,0.009796,-0.005649,0.009992,-0.003769,0.000905,0.021713,0.001782,0.011313
11275,HG_FLU_I565T,I565T,-1.935653,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005378,-0.021973,0.009860,-0.013183,-0.008407,0.000603,...,0.012410,-0.006418,0.010039,-0.005747,0.009982,-0.003956,0.001069,0.021883,0.001647,0.011321
11276,HG_FLU_I565W,I565W,-1.995298,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005377,-0.022060,0.009942,-0.013030,-0.008545,0.000527,...,0.012440,-0.006261,0.009840,-0.005694,0.009949,-0.003873,0.000883,0.021755,0.001847,0.011348
11277,HG_FLU_I565V,I565V,-1.414367,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,-0.005328,-0.021893,0.009898,-0.013199,-0.008618,0.000560,...,0.012248,-0.006273,0.009869,-0.005623,0.009996,-0.003893,0.001010,0.021884,0.001836,0.011473


In [10]:
r2s_train, maes_train, rmses_train, r2s_test, maes_test, rmses_test, rhos_train, rhos_test, folds, num_nonzero_coefs = run_regression(features, target)

print(f"Results:  r2_train: {np.mean(r2s_train):.2f}, r2_test: {np.mean(r2s_test):.2f}, Num coefs: {np.mean(num_nonzero_coefs):.2f}")

Results:  fold 0, r2_train: 0.333, r2_test: 0.325, Num coefs: 83
Results:  fold 1, r2_train: 0.343, r2_test: 0.319, Num coefs: 85
Results:  fold 2, r2_train: 0.334, r2_test: 0.336, Num coefs: 79
Results:  fold 3, r2_train: 0.349, r2_test: 0.326, Num coefs: 89
Results:  fold 4, r2_train: 0.359, r2_test: 0.345, Num coefs: 104
Results:  r2_train: 0.34, r2_test: 0.33, Num coefs: 88.00
Results:  r2_train: 0.34, r2_test: 0.33, Num coefs: 88.00


In [31]:
class FastaDataLoader:
    """
    Data loader for reading a FASTA file and creating batches based on a token limit.
    
    Args:
    - fasta_file (str): Path to the FASTA file.
    - batch_token_limit (int, optional): Maximum number of tokens per batch. Defaults to 4096.
    - model (object): Model object with a `_tokenize` method for tokenizing sequences.
    """
    def __init__(self, fasta_file, batch_token_limit=4096):
        self.fasta_file = fasta_file
        self.batch_token_limit = batch_token_limit
        self.sequences = list(SeqIO.parse(fasta_file, "fasta"))
        self.total_sequences = len(self.sequences)
        
        # Check for duplicate sequence labels
        sequence_labels = [seq.id for seq in self.sequences]
        assert len(set(sequence_labels)) == len(sequence_labels), "Found duplicate sequence labels"

    def __len__(self):
        # Approximate total number of batches
        total_tokens = sum(len(str(seq.seq)) + 2 for seq in self.sequences)  # +2 for BOS and EOS tokens
        return (total_tokens + self.batch_token_limit - 1) // self.batch_token_limit

    def __iter__(self):
        ids, lengths, seqs = [], [], []
        current_token_count = 0

        for seq in self.sequences:
            seq_length = len(seq.seq)
            token_count = seq_length + 2  # Include BOS and EOS tokens
            if current_token_count + token_count > self.batch_token_limit and ids:
                # Yield current batch if adding the new sequence exceeds the token limit
                yield ids, lengths, seqs
                ids, lengths, seqs = [], [], []
                current_token_count = 0

            # Add the current sequence to the batch
            ids.append(seq.id)
            lengths.append(seq_length)
            seqs.append(str(seq.seq))
            current_token_count += token_count

        # Yield any remaining sequences
        if ids:
            yield ids, lengths, seqs

In [34]:
from Bio import SeqIO
from tqdm import tqdm
fasta_file = "../test.fasta"
data_loader = FastaDataLoader(fasta_file)
    
for batch_ids, batch_lengths, batch_seqs in tqdm(data_loader, desc="Processing batches", leave=False):
    print(f"Batch IDs: {batch_ids}")
    print(f"Batch Lengths: {batch_lengths}")
    print(f"Batch Sequences: {batch_seqs}")
    # Here you can process the batch with your model

Batch IDs: ['a4', 'a6', 'a10']
Batch Lengths: [4, 6, 10]
Batch Sequences: ['AAAA', 'AAAAAA', 'AAAAAAAAAA']


# Split datasets by site

In [39]:
import pandas as pd
import random

In [3]:
data = pd.read_csv('../data/nonviral/metadata/PABP_YEAST_Fields2013_doubles_metadata.csv', index_col=0)
data

,ID,mutant,target,sequence
0,PABP_DOUBLES_G169V:F170V,G169V:F170V,0.045765,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
1,PABP_DOUBLES_G169A:F170I,G169A:F170I,0.075799,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
2,PABP_DOUBLES_G169C:F170Y,G169C:F170Y,0.700485,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
3,PABP_DOUBLES_G169A:F170S,G169A:F170S,0.061518,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
4,PABP_DOUBLES_G169Q:F170L,G169Q:F170L,0.036095,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
...,...,...,...,...
36516,PABP_DOUBLES_K164N:E175D,K164N:E175D,0.916383,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
36517,PABP_DOUBLES_K164N:E174D,K164N:E174D,0.887493,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
36518,PABP_DOUBLES_K164T:E174D,K164T:E174D,0.902636,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...
36519,PABP_DOUBLES_K164N:F173L,K164N:F173L,0.637755,MADITDKTAEQLENLNIQDDQKQAATGSESQSVENSSASLYVGDLE...


In [36]:
data['mutant'].str.extract(r'(\d+)').dropna()

,0
0,169
1,169
2,169
3,169
4,169
...,...
36516,164
36517,164
36518,164
36519,164


In [5]:
df = pd.read_csv('../data/nonviral/metadata/PTEN_HUMAN_Fowler2018_metadata.csv', index_col=0)
df

,ID,mutant,target,sequence
0,PTEN_HUMAN_T2A,T2A,0.778594,MAAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
2,PTEN_HUMAN_T2D,T2D,1.173809,MDAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
3,PTEN_HUMAN_T2E,T2E,0.668271,MEAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
5,PTEN_HUMAN_T2G,T2G,0.998143,MGAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
8,PTEN_HUMAN_T2K,T2K,0.896138,MKAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
...,...,...,...,...
8185,PTEN_HUMAN_V403S,V403S,1.032266,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
8186,PTEN_HUMAN_V403T,V403T,0.974582,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
8187,PTEN_HUMAN_V403V,V403V,0.722233,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...
8190,PTEN_HUMAN_V403Y,V403Y,1.170163,MTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVY...


In [ ]:
sites = df['mutant'].str.extract(r'(\d+)').dropna()
sites[0].unique().astype(int)

array([  2,   3,   4,   5,   6,   7,   9,  10,  11,  12,  13,  14,  15,
        16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,
        29,  30,  31,  35,  36,  37,  38,  39,  40,  43,  44,  45,  46,
        47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,
        60,  61,  62,  63,  64,  65,  66,  67,  68,  69,  71,  72,  73,
        74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,
        87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98, 100,
       102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115,
       116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128,
       129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 170,
       171, 172, 174, 175, 176, 177, 178, 179, 182, 184, 186, 187, 188,
       189, 191, 192, 193, 194, 195, 196, 197, 201, 202, 203, 20

In [53]:
df = pd.read_csv('../data/nonviral/metadata/TIM_SULSO_metadata.csv', index_col=0)
print(df.shape)
train_pct=0.8
test_pct=0.2
seed = 42

df["site"] = [int(s[1:-1]) for s in df["mutant"]]
sites = df["site"].unique()
random.seed(seed)
random.shuffle(sites)

if train_pct + test_pct != 1:
    print("Split percentages must sum to 1")
  
df_size = df.shape[0]
df_test_size = df_size*test_pct
test_sites, train_sites = [], []

print(df_size, df_test_size)
# determine sites for test, then train
for site in sites:
    if len(test_sites) <= df_test_size:
        test_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])
    else:
        train_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])


# subset df for train, test data
train_df = df[df["site"].isin(set(train_sites))]
test_df = df[df["site"].isin(set(test_sites))]
test_df

(1519, 4)
1519 303.8


,ID,mutant,target,sequence,site
151,TIM_SULSO_Y52A,Y52A,-1.018648,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,52
152,TIM_SULSO_Y52C,Y52C,-0.487984,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,52
153,TIM_SULSO_Y52D,Y52D,-1.008906,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,52
154,TIM_SULSO_Y52E,Y52E,-0.843564,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,52
155,TIM_SULSO_Y52F,Y52F,0.223149,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,52
...,...,...,...,...,...
1381,TIM_SULSO_N228S,N228S,0.196636,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,228
1382,TIM_SULSO_N228T,N228T,-0.519253,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,228
1383,TIM_SULSO_N228V,N228V,-0.992116,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,228
1384,TIM_SULSO_N228W,N228W,-1.295807,MPRYLKGWLKDVVQLSLRRPSFRASRQRPIISLNERILEFNKRNIT...,228


# Split datasets by site

In [6]:
import pandas as pd
import os


In [36]:
folder = '../data/viral/metadata/'

cols = ['ID','mutant','num_mutations','target','sequence']
for file in os.listdir(folder):
    if file.endswith('_metadata.csv'):
        df = pd.read_csv(os.path.join(folder, file))
        dts_name = file.split('_metadata.csv')[0]
        #length = [len(x.split(':')) for x in df['ID']]
        #print(f"{dts_name}: max length: {max(length)}")
        df['num_mutations'] = [len(x.split(':')) for x in df['mutant']]
        ID = dts_name.split('_')
        df['ID'] = [f"{ID[0]}_{ID[1]}_{i+1}" for i in range(len(df))]
        file_path = os.path.join(folder, dts_name + '.csv')
        df[cols].to_csv(file_path, index=False)
        print(f"Saving {file_path}")
        

df[cols]

Saving ../data/viral/metadata/LASSA_GP_Carr.csv
Saving ../data/viral/metadata/CVB3_3D_Alvarez.csv
Saving ../data/viral/metadata/IAV_H5_HA_Dadonaite.csv
Saving ../data/viral/metadata/PESV_POLG_Tsuboyama.csv
Saving ../data/viral/metadata/SARS2_BA1_SPIKE_Dadonaite.csv
Saving ../data/viral/metadata/HIV1_HV1B9_ENV_DuenasDecamp.csv
Saving ../data/viral/metadata/SARS2_DELTA_SPIKE_Dadonaite.csv
Saving ../data/viral/metadata/CVB3_3B_Alvarez.csv
Saving ../data/viral/metadata/SARS2_XBB15_RBD_Taylor.csv
Saving ../data/viral/metadata/SARS2_RBD_Starr_binding.csv
Saving ../data/viral/metadata/CVB3_2C_Alvarez.csv
Saving ../data/viral/metadata/SARS2_PRD0038_RBD_Starr.csv
Saving ../data/viral/metadata/DENV_POLG_Suphatrakul.csv
Saving ../data/viral/metadata/SARS2_RBD_Starr_expression.csv
Saving ../data/viral/metadata/IAV_H3_NP_Doud.csv
Saving ../data/viral/metadata/BPP22_COAT_Tsuboyama.csv
Saving ../data/viral/metadata/IAV_H1_HA_Doud.csv
Saving ../data/viral/metadata/CVB3_2B_Alvarez.csv
Saving ../data/vi

,ID,mutant,num_mutations,target,sequence
0,BP434_RPC1_1,A18C,1,-0.167281,SISSRVKSKRIQLGLNQCELAQKVGTTQQSIEQLENGKTKRPRFLP...
1,BP434_RPC1_2,A18D,1,-0.238248,SISSRVKSKRIQLGLNQDELAQKVGTTQQSIEQLENGKTKRPRFLP...
2,BP434_RPC1_3,A18E,1,-0.034313,SISSRVKSKRIQLGLNQEELAQKVGTTQQSIEQLENGKTKRPRFLP...
3,BP434_RPC1_4,A18F,1,0.121522,SISSRVKSKRIQLGLNQFELAQKVGTTQQSIEQLENGKTKRPRFLP...
4,BP434_RPC1_5,A18G,1,-0.490995,SISSRVKSKRIQLGLNQGELAQKVGTTQQSIEQLENGKTKRPRFLP...
...,...,...,...,...,...
1454,BP434_RPC1_1455,W58R,1,-3.041730,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1455,BP434_RPC1_1456,W58S,1,-2.247879,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1456,BP434_RPC1_1457,W58T,1,-2.544818,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1457,BP434_RPC1_1458,W58V,1,-2.341668,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...


In [14]:
df = pd.read_csv('../data/marks_data/viral_metadata/AAV2_CAPSD_Sinai_metadata.csv', index_col=0)
df['num_muts'] = [len(x.split(':')) for x in df['ID']]
df.query('num_muts == 28')

,ID,mutant,target,sequence,num_muts
1794,D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568...,D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568...,-3.708075,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28
2988,D561H:E562I:E563D:E564N:I565C:R566C:T567A:T568...,D561H:E562I:E563D:E564N:I565C:R566C:T567A:T568...,-3.832230,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28
6856,D561S:E562V:E563F:E564V:I565V:R566N:T567A:T568...,D561S:E562V:E563F:E564V:I565V:R566N:T567A:T568...,-3.224306,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28


In [16]:
df.iloc[1794]['ID']

'D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568I:N569L:P570F:V571Y:A572M:T573H:E574C:Q575C:Y576Q:G577F:S578I:V579L:S580Q:T581G:N582T:L583V:Q584L:R585I:G586S:N587A:R588H'

In [1]:
def split_data(df, seed, train_pct=0.8, val_pct=0.2):
    """
    This function randomly splits a dataframe into train, validationi data given a seed by mutation site.
    Parameters:
     - df (DataFrame): dataframe containing information about mutants. Mutants should be in the order wt amino acid, site of mutation, mutant amino acid. ex "M1F"
     - seed (int): the seed to be used when shuffling sites randomly.
     - train_pct (float): the percentage of data that will be split into the train dataset. Default is 0.8
     - val_pct (float): the percentage of data that will be split into the validation dataset. Default is 0.2

    Returns:
     - train_df (DataFrame): the DataFrame containing selected data by site to be used as the train dataset.
     - val_df (DataFrame): the DataFrame containing selected data by site to be used as the validation dataset.
    """
    # find sites of mutation and order randomly
    df["site"] = [int(s[1:-1]) for s in df["mutant"]]
    sites = df["site"].unique()
    random.seed(seed)
    random.shuffle(sites)

    if train_pct + val_pct != 1:
        print("Split percentages must sum to 1")
        return

    df_size = df.shape[0]
    df_val_size = df_size*val_pct
    val_sites, train_sites = [], []

    # determine sites for validation, then train
    for site in sites:
        if len(val_sites) <= df_val_size:
            val_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])
        else:
            train_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])

    # subset df for train, test data
    train_df = df[df["site"].isin(set(train_sites))]
    val_df = df[df["site"].isin(set(val_sites))]

    return train_df, val_df